In [1]:
import torch
import torch.nn as nn


In [2]:
a = [2.0, 1.0]      # 원본
b = [4.0, 2.0]      # 같은 방향
c = [-1.0, 2.0]     # 직교
d = [-4.0, -2.0]    #반대방향

같은방향 = (torch.tensor(a), torch.tensor(b))
직교 = (torch.tensor(a), torch.tensor(c))
반대방향 = (torch.tensor(a), torch.tensor(d))

In [3]:
# 내적 했을 때 양수 , 0, 음수 -> 궁합 점수
# 내적은 방향성을 확인하는 점수, 채점표가 된다.
# 파라미터 학습을 하지 않아도 가능. 행렬 곱 한번으로 방향성 확인 가능.
print(torch.dot(같은방향[0], 같은방향[1]))  # 방향 양수
print(torch.dot(직교[0], 직교[1]))         # 0
print(torch.dot(반대방향[0], 반대방향[1]))  # 방향 음수

tensor(10.)
tensor(0.)
tensor(-10.)


In [4]:
# (6, 65)의 벡터 -> 내적
L, d = 6, 64
X_raw = torch.randn(L, d)

In [5]:
import math
# 전치행렬로 행렬곱
점수_raw = X_raw @ X_raw.T  # (6, 64) @ (6, 64) -> (6, 64) @ (64, 6)

어텐션_격자_raw = torch.softmax(점수_raw / math.sqrt(d), dim=1)

In [6]:
# 대각선에 점수가 많이 매겨짐 -> 자기 자신을 중요하게 봄.
# 대각선이 아닌 경우 점수가 낮음
어텐션_격자_raw

tensor([[9.9551e-01, 1.2974e-03, 1.7567e-03, 1.1812e-03, 1.0988e-04, 1.4908e-04],
        [2.7369e-03, 9.8280e-01, 7.2558e-04, 3.0641e-03, 4.4098e-03, 6.2644e-03],
        [7.8846e-04, 1.5437e-04, 9.9840e-01, 8.7494e-05, 3.4636e-04, 2.2725e-04],
        [1.0076e-03, 1.2390e-03, 1.6629e-04, 9.9651e-01, 9.9452e-04, 8.1620e-05],
        [1.5636e-04, 2.9745e-03, 1.0981e-03, 1.6589e-03, 9.9314e-01, 9.7101e-04],
        [2.3049e-04, 4.5909e-03, 7.8277e-04, 1.4792e-04, 1.0550e-03, 9.9319e-01]])

In [7]:
Wq_t = nn.Linear(d, d, bias=False)
Wk_t = nn.Linear(d, d, bias=False)

In [8]:
torch.softmax(Wq_t(X_raw) @ Wk_t(X_raw).T / math.sqrt(d), dim=-1)

tensor([[0.2199, 0.1699, 0.1215, 0.1188, 0.2264, 0.1435],
        [0.1877, 0.1380, 0.2155, 0.1434, 0.1258, 0.1896],
        [0.1805, 0.2238, 0.1344, 0.0860, 0.1438, 0.2316],
        [0.1993, 0.1868, 0.2033, 0.1219, 0.1579, 0.1308],
        [0.1689, 0.1842, 0.2006, 0.0930, 0.1039, 0.2494],
        [0.1081, 0.1890, 0.1885, 0.1486, 0.1439, 0.2218]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
# 셀프 어텐션
class SelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Wq = nn.Linear(d, d, bias=False)   # Q 찾는 것
        self.Wk = nn.Linear(d, d, bias=False)   # K 이름
        self.Wv = nn.Linear(d, d, bias=False)   # V 내용
        self.Wo = nn.Linear(d, d, bias=False)   # O 출력
        
    def forward(self, x):
        Q = self.Wq(X)
        K = self.Wk(X)
        V = self.Wv(X)
        
        d_k = K.size(-1)    # k의 차원 수
        scores = Q @ K.T    # 점수 Q와 K의 전치행렬 합성곱
        scores = scores / math.sqrt(d_k)
        
        # 가중치 : softmax 통과한 것
        weights = torch.softmax(scores, dim=-1)
        
        # 최종 출력 V 행렬 곱
        out = weights @ V
        
        result = self.Wo(out)
        
        return self.Wo(out), weights

In [16]:
TOKEN = ["나는", "배를", "먹었다", "."]
X = torch.tensor([
    [1, 1, 1, 0],   # 나는
    [0, 1, 1, 0],   # 배를
    [1, 0, 0, 1],   # 먹었다
    [0, 0, 1, 1]    # .
], dtype = torch.float)

In [17]:
WQ = torch.tensor([[1, 0, 0, 1], [1, 0, 1, 0], [1, 0, 0, 0], [0, 1, 1, 0]], dtype=torch.float)
WK = torch.tensor([[0, 1, 1, 0], [0, 1, 1, 1], [0, 1, 0, 0], [1, 0, 1, 0]], dtype=torch.float)
WV = torch.tensor([[0, 1, 1, 0], [1, 0, 0, 0], [0, 0, 0, 1], [1, 0, 1, 0]], dtype=torch.float)
WO = torch.eye(4)

In [18]:
WO

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])

In [23]:
model = SelfAttention(4)
with torch.no_grad():
    model.Wq.weight.copy_(WQ.T)
    model.Wk.weight.copy_(WK.T)
    model.Wv.weight.copy_(WV.T)
    model.Wo.weight.copy_(WO.T)
model(X)

(tensor([[1.0000, 0.6225, 1.3535, 0.5449],
         [1.0000, 0.6225, 1.3535, 0.5449],
         [1.0000, 0.6983, 1.0000, 0.8122],
         [1.0000, 0.6859, 1.1019, 0.7411]], grad_fn=<MmBackward0>),
 tensor([[0.1674, 0.1015, 0.4551, 0.2760],
         [0.1674, 0.1015, 0.4551, 0.2760],
         [0.5105, 0.1878, 0.1878, 0.1139],
         [0.4269, 0.1571, 0.2589, 0.1571]], grad_fn=<SoftmaxBackward0>))